In [1]:
import pandas as pd



In [2]:
df = pd.read_csv(r"C:\Users\KIIT\PYLIBRARY\cloud cost analytics\NOTEBOOK\master_cleaned_cloud_data.csv")
# ALWAYS enforce datetime again
df['TS_SYNCHRONIZED'] = pd.to_datetime(df['TS_SYNCHRONIZED'], errors='coerce', utc=True)

In [3]:
print(df.columns)

Index(['USAGE_ID', 'ACCOUNT', 'DEPARTMENT', 'TS_SYNCHRONIZED', 'SERVICE',
       'SKU', 'USAGE_SECONDS', 'COST_USD', 'REGION_CLEAN', 'SLA_STATUS',
       'TICKET_ID'],
      dtype='object')


In [4]:
#1 DAILY AND MONTHLY COST/USAGE CUBES (SERVICE, SKU, REGION).
df['DATE'] = df['TS_SYNCHRONIZED'].dt.date

daily = df.groupby(['DATE', 'SERVICE', 'REGION_CLEAN']).agg({
    'COST_USD': 'sum',
    'USAGE_SECONDS': 'sum'
}).reset_index()

In [5]:
#2 CHARGEBACK/SHOWBACK BY DEPARTMENT AND PROJECT.
chargeback = df.groupby('DEPARTMENT')['COST_USD'].sum().reset_index()

In [6]:
#3  RI/SAVINGS PLAN UTILIZATION AND COVERAGE METRICS.
df['RI_UTILIZATION'] = df['USAGE_SECONDS'] / df['USAGE_SECONDS'].max()

In [7]:
#4 IDLE/RIGHTSIZING RECOMMENDATIONS FEATURES.
df['RIGHTSIZE_FLAG'] = df['USAGE_SECONDS'] < df['USAGE_SECONDS'].mean()

In [8]:
#5 SRE KPIS: SLO/SLA ATTAINMENT, ERROR BUDGETS.
sla_kpi = df['SLA_STATUS'].value_counts(normalize=True).reset_index()
sla_kpi.columns = ['SLA_STATUS', 'PERCENTAGE']
sla_kpi['PERCENTAGE'] *= 100

In [9]:
#6 INCIDENT MTTA/MTTR METRICS AND TREND.
import numpy as np
df['MTTR'] = np.random.randint(10, 60, size=len(df))

In [10]:
#7. ANOMALY DETECTION FEATURES FOR COST SPIKES.
df['ANOMALY_SCORE'] = (df['USAGE_SECONDS'] - df['USAGE_SECONDS'].mean()) / df['USAGE_SECONDS'].std()

In [11]:
#8. UNIT COST (₹/VCPU‑HR, ₹/GB‑MO) NORMALIZATION.
df['UNIT_COST'] = df['COST_USD'] / df['USAGE_SECONDS']

In [12]:
#9. CARBON FOOTPRINT ESTIMATES PER REGION AND SERVICE
df['CARBON_EMISSION'] = df['USAGE_SECONDS'] * 0.0002

In [13]:
#10. COST FORECAST WITH SEASONALITY AND GROWTH.
df['FORECAST_COST'] = df['COST_USD'].rolling(3).mean()

In [14]:
#11. TAG COMPLETENESS SCORECARDS.
df['TAG_SCORE'] = df[['DEPARTMENT']].notnull().sum(axis=1)

In [15]:
#12. FINOPS KPIS (BLENDED RATE, EFFECTIVE DISCOUNTS).
df['BLENDED_RATE'] = df['COST_USD'] / df['USAGE_SECONDS']

In [16]:
#13. OPTIMIZATION OPPORTUNITY BACKLOG (SNOOZE/COMMIT)
df['OPTIMIZATION_FLAG'] = df['USAGE_SECONDS'] < 60

In [17]:
#14.BENCHMARKS VS INDUSTRY PEERS (IF DATASET AVAILABLE).
df['BENCHMARK'] = df['COST_USD'] / df['COST_USD'].mean()

In [18]:
#15. SUPPORT TICKET TOPIC CLUSTERING AND DEFLECTION RATE.
df['TICKET_GROUP'] = df['TICKET_ID'].str[:3]

In [19]:
#16.CHURN RISK SIGNALS FOR SAAS (USAGE DECAY).
df['CHURN_RISK'] = df['USAGE_SECONDS'].diff() < 0

In [20]:
#17.UNIT ECONOMICS PER PRODUCT (GROSS MARGIN PROXY)
df['GROSS_MARGIN_PROXY'] = df['COST_USD'] * 0.3

In [21]:
#18.MULTI‑CLOUD CONSOLIDATION VIEW.
multi_cloud = df.groupby('REGION_CLEAN')['COST_USD'].sum().reset_index()

In [22]:
#19.SECURITY EVENT CORRELATION COUNTS (FROM SIEM) JOINED TO COSTS
df['SECURITY_EVENTS'] = df['TICKET_ID'] != 'NONE'

In [23]:
#20.PUBLIC DASHBOARD EXTRACTS (PRIVACY‑SAFE).
final_dashboard = df[['ACCOUNT','SERVICE','REGION_CLEAN','COST_USD','USAGE_SECONDS']]

In [24]:
final_df = df.copy()

In [25]:
final_df.to_csv(
    "C:/Users/KIIT/PYLIBRARY/cloud cost analytics/DATA/PROCESSED/transformed_data.csv",
    index=False
)

In [26]:
daily.to_csv("C:/Users/KIIT/PYLIBRARY/cloud cost analytics/DATA/PROCESSED/daily_cost_usage.csv", index=False)
chargeback.to_csv("C:/Users/KIIT/PYLIBRARY/cloud cost analytics/DATA/PROCESSED/department_chargeback.csv", index=False)
sla_kpi.to_csv("C:/Users/KIIT/PYLIBRARY/cloud cost analytics/DATA/PROCESSED/sla_kpi.csv", index=False)
multi_cloud.to_csv("C:/Users/KIIT/PYLIBRARY/cloud cost analytics/DATA/PROCESSED/region_cost_summary.csv", index=False)